# Classification de déchets avec PySpark MLlib

Ce notebook montre comment entraîner un modèle de Machine Learning pour identifier le type de déchet à partir de vos images préalablement traitées par `data_tansformation.py`.

In [8]:
import os
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, array_position
from pyspark.sql.types import StructType, StructField, IntegerType, ArrayType, DoubleType
from datetime import datetime

spark = SparkSession.builder \
    .appName("GarbageClassificationML") \
    .config("spark.executor.memory", "2g") \
    .config("spark.driver.memory", "2g") \
    .getOrCreate()

spark


### Chargement et formatage des données
Keras nécessite que les features soient sous forme de `Veteurs`. Nos pixels sont stockés au format texte dans un .parquet

In [9]:
def load_and_prepare_data(path, split):
    df = spark.read.parquet(path)
    label_col = f"y_{split}"
    feature_col = f"x_{split}"

    df = df.withColumn(
        "label",
        array_position(col(label_col), 1) - 1
    )

    return df.select("label", col(feature_col).alias("features")).dropna()

# Test
train_data = load_and_prepare_data("../data/train_data.parquet", "train")
test_data = load_and_prepare_data("../data/test_data.parquet", "test")
print("Train data :")
train_data.show(5)
print("Test data :")
test_data.show(5)


Train data :
+-----+--------------------+
|label|            features|
+-----+--------------------+
|    4|[[185, 211, 210, ...|
|    4|[[175, 198, 196, ...|
|    4|[[129, 143, 142, ...|
|    4|[[190, 217, 214, ...|
|    4|[[136, 149, 146, ...|
+-----+--------------------+
only showing top 5 rows

Test data :
+-----+--------------------+
|label|            features|
+-----+--------------------+
|    0|[[11, 10, 10, 9, ...|
|    0|[[254, 254, 254, ...|
|    0|[[86, 86, 88, 90,...|
|    0|[[106, 111, 113, ...|
|    0|[[163, 163, 164, ...|
+-----+--------------------+
only showing top 5 rows



In [ ]:
CONFIG = {
    "img_size": 64,
    "batch_size": 64,
    "epochs": 100,
    "learning_rate": 3e-4,
    "weight_decay": 1e-4,
    "label_smoothing": 0.05,
    "seed": 42
}

def data_augmentation():
    data_aug = tf.keras.Sequential([
        tf.keras.layers.RandomFlip("horizontal"),
        tf.keras.layers.RandomRotation(0.04),
        tf.keras.layers.RandomZoom(0.05)
    ], name="data_augmentation")
    return data_aug


def compile_model(model):
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=CONFIG["learning_rate"]),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model

def spark_to_numpy(df):

    pdf = df.toPandas()

    
    y = pdf["label"].to_numpy(dtype=np.int32)

    
    X = np.array(pdf["features"].tolist(), dtype=np.float32)

    X = X.reshape(X.shape + (1,))

    return X, y

def save_model(model, name):
    filename = f"{name}.keras"
    save_path = os.path.join("..", "models", filename)
    model.save(save_path)
    print(f"Modèle sauvegardé : {save_path}")

def create_cnn(input_shape, num_classes):
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=input_shape),
        data_augmentation(),
        tf.keras.layers.Rescaling(1.0 / 255.0),
        tf.keras.layers.Conv2D(32, 3, activation="relu", padding="same"),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Conv2D(32, 3, activation="relu", padding="same"),
        tf.keras.layers.MaxPooling2D(),
        tf.keras.layers.Dropout(0.20),
        tf.keras.layers.Conv2D(64, 3, activation="relu", padding="same"),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.MaxPooling2D(),
        tf.keras.layers.Dropout(0.30),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(128, activation="relu"),
        tf.keras.layers.Dropout(0.40),
        tf.keras.layers.Dense(num_classes, activation="softmax")
    ], name="CNN")
    
    return compile_model(model)


def main():
    tf.keras.utils.set_random_seed(CONFIG["seed"])

    train_data = load_and_prepare_data("../data/train_data.parquet", "train")
    test_data = load_and_prepare_data("../data/test_data.parquet", "test")


    X_train, y_train = spark_to_numpy(train_data)
    X_val, y_val = spark_to_numpy(test_data)

    input_shape = X_train.shape[1:]
    print(f"Train: {X_train.shape}, Val: {X_val.shape}")
    print(f"Distribution des images sur les labels: {np.bincount(y_val)}")

    num_classes = int(max(y_train.max(), y_val.max()) + 1)

    class_counts = np.bincount(y_train, minlength=num_classes)
    class_weight = {
        i: float(len(y_train) / (num_classes * count))
        for i, count in enumerate(class_counts) if count > 0
    }

    model = create_cnn(input_shape, num_classes)

    model.summary()

    #Entraînement
    log_name = f"{model.name}_{datetime.now().strftime('%Y%m%d-%H%M%S')}"
    current_log_path = os.path.join("..", "models", "logs", log_name)

    model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=CONFIG["epochs"],
        batch_size=CONFIG["batch_size"],
        class_weight=class_weight,
        callbacks=[
            tf.keras.callbacks.EarlyStopping(
                monitor="val_accuracy", patience=8, restore_best_weights=True, mode="max"
            ),
            tf.keras.callbacks.TensorBoard(log_dir=current_log_path)
        ],
        verbose=2
    )
    val_accuracy = model.evaluate(X_val, y_val, verbose=0)
    save_model(model, f"final_{model.name}_val_acc_{val_accuracy:.4f}")
    spark.catalog.clearCache()


main()


Train: (7324, 64, 64, 1), Val: (3140, 64, 64, 1)
Distribution des images sur les labels: [666 433 795 374 523 349]


E0000 00:00:1774520626.000646   10250 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Model: "CNN"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ data_augmentation (Sequential)  │ (None, 64, 64, 1)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ rescaling (Rescaling)           │ (None, 64, 64, 1)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 64, 64, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 64, 64, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 64, 64, 32)     │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 32, 32, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 32, 32, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 32, 32, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 32, 32, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 16, 16, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 16, 16, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 16384)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │     2,097,280 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 6)              │           774 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,126,502 (8.11 MB)

 Trainable params: 2,126,310 (8.11 MB)

 Non-trainable params: 192 (768.00 B)

Epoch 1/60


W0000 00:00:1774520626.473925   10250 cpu_allocator_impl.cc:82] Allocation of 119996416 exceeds 10% of free system memory.


115/115 - 69s - 597ms/step - accuracy: 0.3078 - loss: 1.8615 - val_accuracy: 0.1191 - val_loss: 5.6036
Epoch 2/60
115/115 - 57s - 491ms/step - accuracy: 0.3353 - loss: 1.6844 - val_accuracy: 0.1191 - val_loss: 9.0409
Epoch 3/60
115/115 - 64s - 553ms/step - accuracy: 0.3493 - loss: 1.6629 - val_accuracy: 0.1252 - val_loss: 9.9199
Epoch 4/60
115/115 - 70s - 612ms/step - accuracy: 0.3641 - loss: 1.6248 - val_accuracy: 0.2182 - val_loss: 5.4011
Epoch 5/60
115/115 - 66s - 571ms/step - accuracy: 0.3669 - loss: 1.6106 - val_accuracy: 0.3022 - val_loss: 3.2587
Epoch 6/60
115/115 - 65s - 563ms/step - accuracy: 0.3520 - loss: 1.5891 - val_accuracy: 0.3602 - val_loss: 2.0038
Epoch 7/60
115/115 - 69s - 601ms/step - accuracy: 0.3532 - loss: 1.5673 - val_accuracy: 0.3761 - val_loss: 1.7153
Epoch 8/60
115/115 - 66s - 573ms/step - accuracy: 0.3711 - loss: 1.5494 - val_accuracy: 0.3497 - val_loss: 1.8991
Epoch 9/60
115/115 - 76s - 663ms/step - accuracy: 0.3890 - loss: 1.5330 - val_accuracy: 0.3449 - va